<a href="https://colab.research.google.com/github/LMS4681/CNN-RL/blob/main/AllocRL/Colab_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AllocRL — Colab 학습 노트북

CNN 공간 인식 + 블록-집합 attention(미래 lookahead) 기반 블록 배치 강화학습.

**자동 저장 / 자동 이어학습**
- 학습 중 `CHECKPOINT_FREQ` step마다 중간 체크포인트 자동 저장
- 학습 종료 시 최종 모델 자동 저장
- 재실행/재접속 시 `AUTO_RESUME=True`면 기존 모델에서 **자동 이어학습**
- `OUTPUT_DIR`을 Google Drive 경로로 두면 결과가 세션 종료 후에도 유지 → **별도 저장 셀 불필요**

**사용법:** 하이퍼파라미터는 **하이퍼파라미터 셀만 수정**하고 **학습 실행 셀을 다시 실행**하면 반영됩니다.

주요 신규 옵션: `--extractor block-attn`, `--n-future-blocks N`, `--auto-resume`, `--checkpoint-freq N`

⚠️ 추출기/관측 구조(`EXTRACTOR`, `N_FUTURE_BLOCKS`, `GRID_SIZE` 등)를 바꾸면 같은 `OUTPUT_DIR`로 이어학습할 수 없습니다 → 새 `OUTPUT_DIR`을 쓰세요.


### 1. 최신 코드 클론 (재실행해도 항상 최신)

In [1]:
%cd /content
import os, shutil
if os.path.exists('/content/CNN-RL'):
    shutil.rmtree('/content/CNN-RL')      # 재실행 시 최신 반영 위해 재클론
!git clone https://github.com/LMS4681/CNN-RL.git
%cd /content/CNN-RL/AllocRL
!ls

/content
Cloning into 'CNN-RL'...
remote: Enumerating objects: 153, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 153 (delta 67), reused 118 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (153/153), 920.25 KiB | 3.08 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/CNN-RL/AllocRL
alloc_env
analyze_blocks.py
Colab_train.ipynb
data
diagnose_synthetic_data.py
export_onnx.py
plot_training_curves.py
requirements.txt
smoke_test.py
test_block_attention_and_future_obs.py
test_diagnostics_and_placement_visualization.py
test_future_block_lookahead.py
test_parallel_training_config.py
test_phase1.py
test_pointer_attention_extractor.py
test_reward.py
test_rl_regressions.py
test_safety_and_workspace_limits.py
test_synthetic.py
test_training_visualization.py
test_train_resume_cli.py
train.py
visualize_eval_placement.py
visualize_grids.py
visualize.py


### 2. 의존성 설치

In [2]:
# Colab엔 torch/numpy/pandas/matplotlib/tensorboard가 이미 있음.
# gymnasium/sb3/sb3-contrib/onnx만 추가 설치되며 torch는 재설치되지 않음.
!pip install -q -r requirements.txt
# ↑ "Restart runtime" 안내가 뜨면: 런타임 재시작 후 셀 1부터 다시 실행.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 49.5 MB/s eta 0:00:00


### 3. Google Drive 마운트 (권장 — 자동 저장/이어학습)

In [3]:
# 결과(체크포인트/모델/로그)를 세션이 끊겨도 유지하려면 Drive 마운트 (권장)
from google.colab import drive
drive.mount('/content/drive')
# 이후 하이퍼파라미터 셀의 OUTPUT_DIR을 /content/drive/... 로 두면
# 자동 저장 + 재접속 시 자동 이어학습이 됩니다. (Drive 미사용 시 이 셀은 건너뛰세요)

Mounted at /content/drive


### 4. (선택) 빠른 검증

In [4]:
# 긴 학습 전 관측/네트워크 정상 동작 점검 (선택)
!python test_future_block_lookahead.py            # 순수 로직 (의존성 불필요)
!python test_block_attention_and_future_obs.py    # 미래 관측 shape + block-attn 추출기
!python smoke_test.py                             # 환경 전체 스모크

.....
----------------------------------------------------------------------
Ran 5 tests in 0.003s

OK
2026-07-08 06:20:56.975383: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
........
----------------------------------------------------------------------
Ran 8 tests in 4.093s

OK
  CNN + Dict Obs 통합 스모크 테스트

[1] 데이터 로드
[DataLoader] 작업장 22개 로드 (지번 총 257개)
[DataLoader] 기배치 444개, 미배치 277개,

### 5. 하이퍼파라미터 ⭐ (이 셀만 수정)

In [7]:
# ===================== 여기만 수정 =====================
# --- 모델 / 관측 구조 ---
EXTRACTOR       = "block-attn"   # "cnn" | "pointer-attn" | "block-attn"
N_FUTURE_BLOCKS = 4              # 미래 lookahead 블록 수 (0=미사용). block-attn엔 3~5 권장
EMBED_DIM       = 64             # attn 토큰 임베딩 차원 (pointer-attn / block-attn 전용)
NUM_HEADS       = 4              # attention head 수 (EMBED_DIM % NUM_HEADS == 0 이어야 함)
FEATURES_DIM    = 256
CNN_OUT_DIM     = 64
GRID_SIZE       = 64             # 점유 그리드 해상도 (메모리 영향)

# --- 학습 하이퍼파라미터 ---
TIMESTEPS   = 100_000            # 이어학습이면 '추가로' 이만큼 더 학습됨
LR          = 3e-4
N_STEPS     = 554
BATCH_SIZE  = 64
N_EPOCHS    = 10
GAMMA       = 1.0

# --- 실행 환경 ---
N_ENVS      = 8                  # 병렬 환경 수 (1이면 단일, >1이면 VEC_ENV 사용)
VEC_ENV     = "auto"             # "auto" | "dummy" | "subproc"
DEVICE      = "auto"             # "auto"면 Colab GPU 자동 사용
N_EVAL      = 5
ACTIVE_WORKSPACE_CODES = "PE052,PE055,PE051,PE050,PE049"  # ""(빈문자열)=전체 작업장

# --- 저장 / 이어학습 ---
# Drive 마운트했다면 Drive 경로로 두면 자동 저장 + 재접속 시 자동 이어학습.
OUTPUT_DIR      = "/content/drive/MyDrive/CNN-RL-outputs/block_attn"
# (Drive 미사용 시: OUTPUT_DIR = "./output_block_attn")
AUTO_RESUME     = True           # OUTPUT_DIR에 호환 모델 있으면 자동 이어학습 (설정 바뀌면 중단)
CHECKPOINT_FREQ = 10_000         # 중간 체크포인트 저장 주기(step). 0=비활성
RESUME_FROM     = ""             # 특정 zip에서 수동 이어학습. 보통 비워두고 AUTO_RESUME 사용
EXPORT_ONNX     = True
# =======================================================
print(f"설정: extractor={EXTRACTOR}, n_future_blocks={N_FUTURE_BLOCKS}, "
      f"timesteps={TIMESTEPS}, auto_resume={AUTO_RESUME}, "
      f"checkpoint_freq={CHECKPOINT_FREQ}, output={OUTPUT_DIR}")

설정: extractor=block-attn, n_future_blocks=4, timesteps=100000, auto_resume=True, checkpoint_freq=10000, output=/content/drive/MyDrive/CNN-RL-outputs/block_attn


### 6. 학습 실행 (자동 체크포인트 + 자동 이어학습)

In [8]:
# 하이퍼파라미터 셀의 값으로 학습 명령을 구성해 실행.
# 값을 바꾸고 이 셀만 다시 실행하면 반영됩니다. (AUTO_RESUME=True면 자동 이어학습)
cmd = (
    "python train.py"
    " --data-dir ./data"
    f" --output-dir {OUTPUT_DIR}"
    f" --timesteps {TIMESTEPS}"
    f" --lr {LR}"
    f" --n-steps {N_STEPS}"
    f" --batch-size {BATCH_SIZE}"
    f" --n-epochs {N_EPOCHS}"
    f" --gamma {GAMMA}"
    f" --grid-size {GRID_SIZE}"
    f" --extractor {EXTRACTOR}"
    f" --n-future-blocks {N_FUTURE_BLOCKS}"
    f" --features-dim {FEATURES_DIM}"
    f" --cnn-out-dim {CNN_OUT_DIM}"
    f" --extractor-embed-dim {EMBED_DIM}"
    f" --extractor-heads {NUM_HEADS}"
    f" --checkpoint-freq {CHECKPOINT_FREQ}"
    f" --n-envs {N_ENVS}"
    f" --vec-env {VEC_ENV}"
    f" --device {DEVICE}"
    f" --n-eval {N_EVAL}"
    f' --active-workspace-codes "{ACTIVE_WORKSPACE_CODES}"'
)
if AUTO_RESUME:
    cmd += " --auto-resume"
if RESUME_FROM:
    cmd += f" --resume-from {RESUME_FROM}"
if not EXPORT_ONNX:
    cmd += " --no-export-onnx"
print(cmd, "\n" + "="*70)
!{cmd}

python train.py --data-dir ./data --output-dir /content/drive/MyDrive/CNN-RL-outputs/block_attn --timesteps 100000 --lr 0.0003 --n-steps 554 --batch-size 64 --n-epochs 10 --gamma 1.0 --grid-size 64 --extractor block-attn --n-future-blocks 4 --features-dim 256 --cnn-out-dim 64 --extractor-embed-dim 64 --extractor-heads 4 --checkpoint-freq 10000 --n-envs 8 --vec-env auto --device auto --n-eval 5 --active-workspace-codes "PE052,PE055,PE051,PE050,PE049" --auto-resume 
2026-07-08 06:22:33.000903: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your so

### 7. TensorBoard

In [ ]:
import os
os.environ["TB_LOGDIR"] = f"{OUTPUT_DIR}/tb_logs"
%load_ext tensorboard
%tensorboard --logdir $TB_LOGDIR

### 8. (선택) 배치 시각화

In [ ]:
# 배치 시각화 (선택). --n-future-blocks 를 학습 때 값과 반드시 동일하게 (관측 공간 일치)!
cmd = (
    "python visualize_eval_placement.py"
    f" --model-path {OUTPUT_DIR}/block_placement_ppo.zip"
    f" --output-dir {OUTPUT_DIR}/eval_visualization"
    f" --grid-size {GRID_SIZE}"
    f" --n-future-blocks {N_FUTURE_BLOCKS}"
    f' --active-workspace-codes "{ACTIVE_WORKSPACE_CODES}"'
)
print(cmd)
!{cmd}

### 9. 저장된 산출물 확인

In [ ]:
# 저장된 산출물 확인 (OUTPUT_DIR이 Drive면 세션 종료 후에도 유지됨)
import os
print("OUTPUT_DIR:", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(" ", f)
ckpt = os.path.join(OUTPUT_DIR, "checkpoints")
print("체크포인트:", sorted(os.listdir(ckpt)) if os.path.isdir(ckpt) else "(없음)")

## 팁

- **자동 이어학습**: `AUTO_RESUME=True` + `OUTPUT_DIR`이 Drive면, 세션이 끊겨도
  다시 셀 1→설치→마운트→학습 순으로 실행하면 마지막 체크포인트에서 이어집니다.
  (별도 저장/복원 셀 불필요.)
- **각 실행 = 추가 학습**: 이어학습 시 `TIMESTEPS`는 '추가로' 더 도는 step 수입니다.
- **A/B 비교**: `(EXTRACTOR, N_FUTURE_BLOCKS, OUTPUT_DIR)`을 바꿔 여러 폴더로 학습하면
  TensorBoard에서 비교됩니다. 예: `("cnn", 0)` vs `("block-attn", 3)` vs `("block-attn", 5)`.
- **설정 변경 시**: 추출기/관측 구조를 바꾸면 같은 `OUTPUT_DIR`로 이어학습 불가(자동 중단).
  새 `OUTPUT_DIR`을 지정하세요.
- `EMBED_DIM % NUM_HEADS == 0` 이어야 attn 추출기가 생성됩니다.
